# Qwen3-TTS raw embedding projection with SEAM

This applies the paper's ordered spectral subtoken merging to the existing LLaVA-to-Qwen experiment. Each unique Qwen token is independently decomposed into LLaVA subtokens, converted to filtered complex DFT features, and aligned to its native Qwen raw embedding with closed-form ridge regression. The 5,500 semantic texts calibrate the map; the 100 MiniMax texts remain the TTS evaluation set.

In [ ]:
!pip install -q -U transformers sentencepiece safetensors huggingface_hub

In [ ]:
import gc
import hashlib
import json
import math
import os
import time
from collections import Counter, defaultdict
from pathlib import Path

import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from IPython.display import FileLink, display
from safetensors import safe_open
from tqdm.auto import tqdm
from transformers import AutoTokenizer

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)
print("Torch:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SOURCE_MODEL_ID = "llava-hf/llava-1.5-7b-hf"


def load_embedding_table(model_id):
    source_tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        use_fast=True,
    )
    index_path = hf_hub_download(
        repo_id=model_id,
        filename="model.safetensors.index.json",
    )
    with open(index_path, "r", encoding="utf-8") as handle:
        index = json.load(handle)
    keys = [
        key
        for key in index["weight_map"]
        if key.endswith("embed_tokens.weight")
    ]
    assert len(keys) == 1, keys
    embedding_key = keys[0]
    embedding_shard = index["weight_map"][embedding_key]
    shard_path = hf_hub_download(
        repo_id=model_id,
        filename=embedding_shard,
    )
    with safe_open(shard_path, framework="pt", device="cpu") as handle:
        weight = handle.get_tensor(embedding_key).contiguous()
    return weight, source_tokenizer, embedding_key, embedding_shard


(
    embedding_weight,
    tokenizer,
    embedding_key,
    embedding_shard,
) = load_embedding_table(SOURCE_MODEL_ID)

assert tokenizer.is_fast
print("Loaded:", SOURCE_MODEL_ID)
print("Representation: raw input embedding lookup")
print("Embedding key:", embedding_key)
print("Embedding shard:", embedding_shard)
print("Embedding table:", embedding_weight.shape, embedding_weight.dtype)

In [ ]:
def find_one(filename):
    matches = list(Path("/kaggle/input").rglob(filename))
    if len(matches) != 1:
        available = [
            str(path)
            for path in Path("/kaggle/input").rglob("*.pt")
        ]
        raise FileNotFoundError(
            f"Expected exactly one {filename!r}, found {len(matches)}.\n"
            + "Available .pt files:\n"
            + "\n".join(available)
        )
    return matches[0]


REQUIRED_KEYS = {
    "embeddings",
    "row_offsets",
    "token_ids",
    "character_offsets",
    "token_strings",
    "prompt_texts",
    "prompt_sha256",
    "wrapped_texts",
    "full_input_ids",
    "source_voices",
    "model_id",
    "target_representation",
}


def load_qwen_bundle(filename):
    path = find_one(filename)
    bundle = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )
    missing = REQUIRED_KEYS - set(bundle)
    assert not missing, f"Missing keys in {filename}: {missing}"
    expected_hashes = [
        hashlib.sha256(text.encode("utf-8")).hexdigest()
        for text in bundle["prompt_texts"]
    ]
    assert expected_hashes == list(bundle["prompt_sha256"])
    return path, bundle


TRAIN_QWEN_PATH, qwen = load_qwen_bundle(
    "qwen3_tts_semantic_5500_raw_text_embeddings.pt"
)
EVAL_QWEN_PATH, eval_qwen = load_qwen_bundle(
    "qwen3_tts_minimax_english_100_raw_text_embeddings.pt"
)

assert qwen["embeddings"].shape[1] == eval_qwen["embeddings"].shape[1]
assert not set(qwen["prompt_texts"]).intersection(eval_qwen["prompt_texts"])
assert qwen.get("benchmark_prompts_used_for_fitting") is False

print("Training file:", TRAIN_QWEN_PATH)
print("Training prompts:", len(qwen["prompt_texts"]))
print("Training token occurrences:", len(qwen["token_ids"]))
print()
print("Evaluation file:", EVAL_QWEN_PATH)
print("Evaluation prompts:", len(eval_qwen["prompt_texts"]))
print("Evaluation token occurrences:", len(eval_qwen["token_ids"]))

## SEAM token decomposition and spectral features

The old character-overlap average is intentionally not used. A Qwen token's exact surface text is retokenized by LLaVA by itself, producing an ordered source-subtoken sequence. Following the paper, the sequence is transformed with an unnormalized real DFT, Gaussian-filtered with `K=4` and `sigma=2`, and zero-padded to a fixed `2*K*d_source` feature vector.

In [ ]:
SEAM_K = 4
SEAM_SIGMA = 2.0
SEAM_LAMBDA = 1e-6
SPECTRAL_BATCH_SIZE = 256


def iter_occurrences(bundle):
    row_offsets = bundle["row_offsets"].long()
    flat_strings = list(bundle["token_strings"])
    for prompt_index, text in enumerate(bundle["prompt_texts"]):
        flat_start = int(row_offsets[prompt_index])
        flat_end = int(row_offsets[prompt_index + 1])
        for flat_index in range(flat_start, flat_end):
            char_start, char_end = (
                bundle["character_offsets"][flat_index].long().tolist()
            )
            surface = text[char_start:char_end]
            if not surface:
                raise ValueError(
                    f"Empty Qwen token surface at prompt {prompt_index}, "
                    f"flat row {flat_index}"
                )
            yield {
                "prompt_index": prompt_index,
                "flat_index": flat_index,
                "token_id": int(bundle["token_ids"][flat_index]),
                "token_string": flat_strings[flat_index],
                "surface": surface,
            }


def source_ids_for_surface(surface):
    source_ids = tokenizer.encode(
        surface,
        add_special_tokens=False,
    )
    if source_ids:
        return tuple(int(value) for value in source_ids), False
    if tokenizer.unk_token_id is None:
        raise ValueError(f"LLaVA produced no subtokens for {surface!r}")
    return (int(tokenizer.unk_token_id),), True


def make_unique_calibration(bundle):
    groups = defaultdict(list)
    for record in iter_occurrences(bundle):
        groups[record["token_id"]].append(record)

    token_ids = []
    source_sequences = []
    targets = []
    surfaces = []
    empty_fallbacks = 0

    for token_id in sorted(groups):
        records = groups[token_id]
        surface_counts = Counter(record["surface"] for record in records)
        canonical_surface = sorted(
            surface_counts,
            key=lambda value: (-surface_counts[value], value),
        )[0]
        representative = next(
            record
            for record in records
            if record["surface"] == canonical_surface
        )
        source_ids, used_fallback = source_ids_for_surface(canonical_surface)
        empty_fallbacks += int(used_fallback)
        token_ids.append(token_id)
        source_sequences.append(source_ids)
        targets.append(bundle["embeddings"][representative["flat_index"]])
        surfaces.append(canonical_surface)

    return (
        torch.tensor(token_ids, dtype=torch.int32),
        source_sequences,
        torch.stack(targets).contiguous(),
        surfaces,
        empty_fallbacks,
    )


def make_occurrence_decompositions(bundle):
    records = list(iter_occurrences(bundle))
    source_sequences = []
    empty_fallbacks = 0
    for record in records:
        source_ids, used_fallback = source_ids_for_surface(record["surface"])
        source_sequences.append(source_ids)
        empty_fallbacks += int(used_fallback)
    return records, source_sequences, empty_fallbacks


@torch.inference_mode()
def build_spectral_features(source_sequences, description):
    source_dimension = int(embedding_weight.shape[1])
    feature_dimension = 2 * SEAM_K * source_dimension
    features = torch.empty(
        len(source_sequences),
        feature_dimension,
        dtype=torch.float16,
    )
    length_groups = defaultdict(list)
    for row_index, source_ids in enumerate(source_sequences):
        length_groups[len(source_ids)].append(row_index)

    total_batches = sum(
        math.ceil(len(rows) / SPECTRAL_BATCH_SIZE)
        for rows in length_groups.values()
    )
    progress = tqdm(total=total_batches, desc=description)

    for sequence_length, row_indices in sorted(length_groups.items()):
        for batch_start in range(0, len(row_indices), SPECTRAL_BATCH_SIZE):
            batch_rows = row_indices[
                batch_start:batch_start + SPECTRAL_BATCH_SIZE
            ]
            ids = torch.tensor(
                [source_sequences[index] for index in batch_rows],
                dtype=torch.long,
            )
            sequence_embeddings = embedding_weight[ids].float()
            spectrum = torch.fft.rfft(
                sequence_embeddings,
                dim=1,
                norm=None,
            )
            effective_bins = min(SEAM_K, sequence_length // 2 + 1)
            frequencies = torch.arange(
                effective_bins,
                dtype=torch.float32,
            )
            gaussian = torch.exp(
                -(frequencies ** 2) / (2 * SEAM_SIGMA ** 2)
            )
            filtered = spectrum[:, :effective_bins] * gaussian[None, :, None]
            packed = torch.zeros(
                len(batch_rows),
                SEAM_K,
                2,
                source_dimension,
                dtype=torch.float32,
            )
            packed[:, :effective_bins, 0] = filtered.real
            packed[:, :effective_bins, 1] = filtered.imag
            features[batch_rows] = packed.flatten(1).to(torch.float16)
            progress.update(1)

    progress.close()
    return features.contiguous()


(
    calibration_qwen_token_ids,
    calibration_source_sequences,
    Y_train_cpu,
    calibration_surfaces,
    train_empty_fallbacks,
) = make_unique_calibration(qwen)

(
    eval_records,
    eval_source_sequences,
    eval_empty_fallbacks,
) = make_occurrence_decompositions(eval_qwen)

X_train_cpu = build_spectral_features(
    calibration_source_sequences,
    "Building SEAM calibration features",
)
X_eval_cpu = build_spectral_features(
    eval_source_sequences,
    "Building SEAM evaluation features",
)
Y_eval_cpu = eval_qwen["embeddings"].float().contiguous()

sequence_to_targets = defaultdict(set)
for token_id, source_ids in zip(
    calibration_qwen_token_ids.tolist(),
    calibration_source_sequences,
):
    sequence_to_targets[source_ids].add(token_id)
collision_groups = sum(
    len(targets) > 1
    for targets in sequence_to_targets.values()
)

train_token_id_set = set(calibration_qwen_token_ids.tolist())
eval_token_ids = eval_qwen["token_ids"].long()
eval_seen_mask = torch.tensor(
    [int(token_id) in train_token_id_set for token_id in eval_token_ids],
    dtype=torch.bool,
)

length_counts = Counter(len(ids) for ids in calibration_source_sequences)
print("Unique Qwen calibration tokens:", len(calibration_qwen_token_ids))
print("SEAM training features:", X_train_cpu.shape)
print("SEAM evaluation features:", X_eval_cpu.shape)
print("Source-subtoken length distribution:", dict(sorted(length_counts.items())))
print("Conflicting source-sequence groups:", collision_groups)
print("Empty-decomposition fallbacks (train/eval):", train_empty_fallbacks, eval_empty_fallbacks)
print("MiniMax token occurrences seen in calibration:", f"{eval_seen_mask.float().mean().item():.1%}")

## Closed-form SEAM calibration

The intercept is handled by centering `X` and `Y`; the dual ridge form avoids forming the much larger `(2*K*d_source) x (2*K*d_source)` matrix. `lambda=1e-6` is attempted first, as in the paper, and is increased only if float32 Cholesky requires it.

In [ ]:
globals().pop("embedding_weight", None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

fit_device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)
fit_start = time.perf_counter()

X_train = X_train_cpu.float().to(fit_device)
Y_train = Y_train_cpu.float().to(fit_device)
X_mean = X_train.mean(dim=0, keepdim=True)
Y_mean = Y_train.mean(dim=0, keepdim=True)
X_train.sub_(X_mean)
Y_centered = Y_train - Y_mean

gram = X_train @ X_train.T
gram = (gram + gram.T) * 0.5

ridge_cholesky = None
lambda_reg = None
for multiplier in [1, 10, 100, 1000, 10000, 100000, 1000000]:
    candidate_lambda = SEAM_LAMBDA * multiplier
    candidate_gram = gram.clone()
    candidate_gram.diagonal().add_(candidate_lambda)
    candidate, info = torch.linalg.cholesky_ex(candidate_gram)
    del candidate_gram
    if int(info.max()) == 0:
        ridge_cholesky = candidate
        lambda_reg = candidate_lambda
        break
    del candidate

assert ridge_cholesky is not None, "SEAM ridge system was not positive definite"
dual_coefficients = torch.cholesky_solve(
    Y_centered,
    ridge_cholesky,
)

target_mean_norm = Y_train.norm(dim=1).mean()


@torch.inference_mode()
def apply_seam(features_cpu, batch_size=256):
    output_parts = []
    for start in tqdm(
        range(0, len(features_cpu), batch_size),
        desc="Applying SEAM projector",
    ):
        end = min(start + batch_size, len(features_cpu))
        features = features_cpu[start:end].float().to(fit_device)
        features.sub_(X_mean)
        kernel_rows = features @ X_train.T
        prediction = kernel_rows @ dual_coefficients + Y_mean
        prediction = F.normalize(prediction, dim=1) * target_mean_norm
        output_parts.append(prediction.cpu())
    return torch.cat(output_parts).contiguous()


Y_train_pred = F.normalize(
    gram @ dual_coefficients + Y_mean,
    dim=1,
) * target_mean_norm
Y_eval_pred = apply_seam(X_eval_cpu)


def cosine_summary(name, predicted, target, mask=None):
    predicted = predicted.float().cpu()
    target = target.float().cpu()
    if mask is not None:
        predicted = predicted[mask]
        target = target[mask]
    if len(predicted) == 0:
        result = {"count": 0, "mean_cosine": None, "median_cosine": None}
    else:
        cosine = F.cosine_similarity(predicted, target, dim=1)
        result = {
            "count": len(cosine),
            "mean_cosine": float(cosine.mean()),
            "median_cosine": float(cosine.median()),
            "minimum_cosine": float(cosine.min()),
            "maximum_cosine": float(cosine.max()),
        }
    print(name, result)
    return result


train_metrics = cosine_summary(
    "Calibration-token diagnostics:",
    Y_train_pred,
    Y_train_cpu,
)
eval_metrics = cosine_summary(
    "MiniMax all-token diagnostics:",
    Y_eval_pred,
    Y_eval_cpu,
)
eval_seen_metrics = cosine_summary(
    "MiniMax seen-token diagnostics:",
    Y_eval_pred,
    Y_eval_cpu,
    eval_seen_mask,
)
eval_unseen_metrics = cosine_summary(
    "MiniMax unseen-token diagnostics:",
    Y_eval_pred,
    Y_eval_cpu,
    ~eval_seen_mask,
)

fit_seconds = time.perf_counter() - fit_start
print("Requested lambda:", SEAM_LAMBDA)
print("Used lambda:", lambda_reg)
print("Target mean norm:", float(target_mean_norm))
print("Fit and projection seconds:", round(fit_seconds, 2))
assert Y_eval_pred.shape == Y_eval_cpu.shape
assert torch.isfinite(Y_eval_pred).all()

In [ ]:
PROJECTOR_PATH = Path(
    "/kaggle/working/qwen3_tts_llava_seam_projector.pt"
)
PROJECTED_PATH = Path(
    "/kaggle/working/"
    "qwen3_tts_minimax_english_100_projected_raw_text_embeddings.pt"
)

calibration_source_flat = []
calibration_source_offsets = [0]
for source_ids in calibration_source_sequences:
    calibration_source_flat.extend(source_ids)
    calibration_source_offsets.append(len(calibration_source_flat))

projector_bundle = {
    "format_version": 1,
    "kind": "seam_spectral_ridge_cross_tokenizer",
    "paper_method": "Spectral Embedding Alignment and Merging",
    "source_model": SOURCE_MODEL_ID,
    "target_model": qwen["model_id"],
    "source_representation": "raw_input_embedding_lookup",
    "target_representation": qwen["target_representation"],
    "token_alignment": "independent_qwen_surface_to_ordered_llava_subtokens",
    "spectral_k": SEAM_K,
    "spectral_sigma": SEAM_SIGMA,
    "requested_lambda": SEAM_LAMBDA,
    "used_lambda": lambda_reg,
    "feature_layout": "frequency_major_real_then_imag_per_bin_zero_padded",
    "calibration_qwen_token_ids": calibration_qwen_token_ids,
    "calibration_surfaces": calibration_surfaces,
    "calibration_source_token_ids": torch.tensor(
        calibration_source_flat, dtype=torch.int32
    ),
    "calibration_source_row_offsets": torch.tensor(
        calibration_source_offsets, dtype=torch.int64
    ),
    "feature_mean": X_mean.cpu().to(torch.float16),
    "target_mean": Y_mean.cpu().to(torch.float16),
    "dual_coefficients": dual_coefficients.cpu().to(torch.float16),
    "target_mean_norm": float(target_mean_norm),
    "norm_matching": "global_mean_native_qwen_l2_norm",
    "training_prompt_count": len(qwen["prompt_texts"]),
    "training_unique_qwen_token_count": len(calibration_qwen_token_ids),
    "training_dataset": "semantic-5500",
    "benchmark_prompts_used_for_fitting": False,
    "training_embedding_diagnostics": train_metrics,
    "fit_seconds": fit_seconds,
}

projected_bundle = {
    "format_version": 2,
    "kind": "qwen3_tts_projected_raw_text_embeddings",
    "projected_text_mode": "native_qwen_content_token_sequence",
    "embeddings": Y_eval_pred.to(torch.float16).contiguous(),
    "native_target_embeddings": eval_qwen["embeddings"].clone(),
    "conditioning_storage": "active_content_tokens_only",
    "row_offsets": eval_qwen["row_offsets"].clone(),
    "token_ids": eval_qwen["token_ids"].clone(),
    "character_offsets": eval_qwen["character_offsets"].clone(),
    "token_strings": list(eval_qwen["token_strings"]),
    "prompt_texts": list(eval_qwen["prompt_texts"]),
    "prompt_sha256": list(eval_qwen["prompt_sha256"]),
    "source_voices": list(eval_qwen["source_voices"]),
    "wrapped_texts": list(eval_qwen["wrapped_texts"]),
    "full_input_ids": list(eval_qwen["full_input_ids"]),
    "source_model": SOURCE_MODEL_ID,
    "target_model": eval_qwen["model_id"],
    "source_representation": "raw_input_embedding_lookup",
    "target_representation": eval_qwen["target_representation"],
    "injection_point": "immediately_before_talker.text_projection",
    "conditioning_layout": "exact_native_qwen_content_positions",
    "projector_kind": projector_bundle["kind"],
    "token_alignment": projector_bundle["token_alignment"],
    "spectral_k": SEAM_K,
    "spectral_sigma": SEAM_SIGMA,
    "evaluation_embedding_diagnostics": eval_metrics,
    "evaluation_seen_token_diagnostics": eval_seen_metrics,
    "evaluation_unseen_token_diagnostics": eval_unseen_metrics,
    "evaluation_seen_token_fraction": float(eval_seen_mask.float().mean()),
    "training_prompt_count": len(qwen["prompt_texts"]),
    "evaluation_prompt_count": len(eval_qwen["prompt_texts"]),
    "benchmark_prompts_used_for_fitting": False,
    "evaluation_validity": "out_of_sample_semantic5500_fit_minimax100_eval",
}

assert projected_bundle["embeddings"].shape == eval_qwen["embeddings"].shape
assert torch.equal(projected_bundle["row_offsets"], eval_qwen["row_offsets"])
assert projected_bundle["prompt_sha256"] == eval_qwen["prompt_sha256"]
assert torch.isfinite(projected_bundle["embeddings"].float()).all()

torch.save(projector_bundle, PROJECTOR_PATH)
torch.save(projected_bundle, PROJECTED_PATH)

saved = torch.load(PROJECTED_PATH, map_location="cpu", weights_only=False)
assert saved["prompt_sha256"] == eval_qwen["prompt_sha256"]
assert torch.equal(saved["row_offsets"], eval_qwen["row_offsets"])

print("Saved projector:", PROJECTOR_PATH)
print("Saved projected embeddings:", PROJECTED_PATH)
print("Projected shape:", saved["embeddings"].shape)
os.chdir("/kaggle/working")
display(FileLink(PROJECTOR_PATH.name))
display(FileLink(PROJECTED_PATH.name))